# 🎬 Build a shareable Trace — your SAE + your prompt → `/observatory/trace`

This notebook turns **your** SAE (on HuggingFace) + **your** favourite prompt into a `trace.json` file
that the [openinterp.org](https://openinterp.org) **Trace Theater** renders as an interactive, shareable
token-by-token feature activation movie.

**Inputs**
- A trained SAE on HF Hub (`sae.safetensors` with `W_enc`, `b_enc`, `W_dec`, `b_dec` — e.g. output of `01`/`02`/`03`).
- A base model on HF Hub (the one the SAE was trained against).
- One prompt you care about.

**Output**
A single `trace.json` matching the `TraceData` TypeScript interface:

```ts
interface TraceData {
  prompt: string
  model: string
  layer: string               // e.g. 'L15 residual'
  sae_repo: string
  tokens: string[]            // generated tokens, leading-space preserved
  features: { id: string; name: string; desc: string; auroc: number }[]  // top-10
  activations: number[][]     // [features][tokens], 0.0 – 1.0 normalised per feature
  counterfactuals: Record<string, Record<string, string>>
}
```

**Runtime budget** ≤ 5 min on a free T4.

**Pipeline**
```
  prompt ─► base model ─► residual @ LAYER (per generated token)
                             │
                             ▼
                          SAE encode
                             │
                             ▼
                    features × tokens matrix
                             │
                             ▼
          pick top-10 by total activation, normalise per-feature
                             │
                             ▼
                        trace.json
```

In [ ]:
# Cell 2 — install pinned deps (no flash-attn; SDPA only)
!pip install -q \
    transformers==4.57.1 \
    accelerate==1.12.0 \
    safetensors==0.4.5 \
    huggingface_hub==1.5.0 \
    tqdm

In [ ]:
# Cell 3 — CONFIG (edit me)
# -----------------------------------------------------------------------------
# Point at the SAE you published (from notebook 01/02/03) + the base model.
# -----------------------------------------------------------------------------

HF_SAE_REPO    = 'your-username/gemma-2-2b-sae-L15'  # <-- your SAE repo on HF
HF_BASE_MODEL  = 'google/gemma-2-2b'                 # base model the SAE was trained on
LAYER          = 15                                  # residual layer index (must match SAE)
D_MODEL        = 2304                                # residual width of the base model
D_SAE          = 16384                               # SAE dictionary size (d_sae)
K              = 32                                  # TopK — leave as-is if SAE was TopK

PROMPT         = 'A 52-year-old patient arrives with chest pain,'
MAX_NEW_TOKENS = 25        # keep <= ~30 for 5-min T4 budget
TOP_N_FEATURES = 10        # shown in the trace (Theater renders 10 lanes)

SAE_FILE       = 'sae.safetensors'           # weight file inside HF_SAE_REPO
CATALOG_FILE   = 'feature_catalog.json'      # optional — from notebook 04
OUTPUT_TRACE   = 'trace.json'                # local filename
UPLOAD_TO_HF   = True                        # push trace.json back to HF_SAE_REPO

print(f'SAE:        {HF_SAE_REPO}')
print(f'Base model: {HF_BASE_MODEL}')
print(f'Layer:      L{LAYER} residual')
print(f'Prompt:     {PROMPT!r}')

## 🔐 Authenticate with HuggingFace

Add your HF token to Colab **Secrets** (🔑 icon, left sidebar) as `HF_TOKEN`.
A read-only token is enough unless you want to upload the trace (`UPLOAD_TO_HF = True`),
in which case you need **write** on `HF_SAE_REPO`.

In [ ]:
# Cell 5 — HF auth via Colab secret
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab userdata')
except Exception as e:
    # Non-Colab / no secret — fall back to env var.
    if 'HF_TOKEN' not in os.environ:
        print('WARN no HF_TOKEN found — public repos will still work, writes will not.')
    else:
        print('HF_TOKEN from environment')

from huggingface_hub import login
if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

In [ ]:
# Cell 6 — load SAE + base model (bf16 + SDPA, no flash-attn)
import torch, torch.nn as nn
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if device == 'cuda' else torch.float32
print(f'device={device}  dtype={dtype}')

# ---- SAE ---------------------------------------------------------------------
sae_path = hf_hub_download(repo_id=HF_SAE_REPO, filename=SAE_FILE)
sae_sd   = load_file(sae_path)

class TopKSAE(nn.Module):
    """Minimal TopK SAE: encode(x) -> (features [B,D_SAE], topk_idx).
    Matches the encode/decode convention used in notebooks 01–04."""
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.k = k

    @torch.no_grad()
    def encode(self, x):
        # x: [tokens, d_model]
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        # TopK sparsification (ReLU-style; preserves magnitude).
        vals, idx = pre.topk(self.k, dim=-1)
        vals = vals.clamp_min(0)
        feats = torch.zeros_like(pre)
        feats.scatter_(-1, idx, vals)
        return feats

sae = TopKSAE(D_MODEL, D_SAE, K).to(device=device, dtype=torch.float32)
missing, unexpected = sae.load_state_dict(sae_sd, strict=False)
if missing:    print('WARN missing SAE keys:   ', missing)
if unexpected: print('WARN unexpected SAE keys:', unexpected)
sae.eval()
print(f'SAE loaded: d_model={D_MODEL} d_sae={D_SAE} k={K}')

# ---- base model + tokenizer -------------------------------------------------
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=dtype,                # transformers 4.57: dtype= not torch_dtype=
    attn_implementation='sdpa', # no flash-attn
    device_map=device,
)
model.eval()

# Locate the transformer stack — supports model.model.layers OR model.language_model.layers.
def get_layers(m):
    if hasattr(m, 'model') and hasattr(m.model, 'layers'):
        return m.model.layers
    if hasattr(m, 'language_model') and hasattr(m.language_model, 'layers'):
        return m.language_model.layers
    if hasattr(m, 'model') and hasattr(m.model, 'language_model') \
       and hasattr(m.model.language_model, 'layers'):
        return m.model.language_model.layers
    raise RuntimeError('Could not locate transformer layers on model')

layers = get_layers(model)
assert 0 <= LAYER < len(layers), f'LAYER={LAYER} out of range (n_layers={len(layers)})'
print(f'Base model loaded: {HF_BASE_MODEL}  n_layers={len(layers)}  hooking L{LAYER}')

In [ ]:
# Cell 7 — try to load feature_catalog.json (from notebook 04); otherwise fall back to ID-only
import json
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import EntryNotFoundError

feature_catalog = None
try:
    cat_path = hf_hub_download(repo_id=HF_SAE_REPO, filename=CATALOG_FILE)
    with open(cat_path) as f:
        raw = json.load(f)
    # Accept either {"features": [...]} or a flat list or {id: {...}}.
    if isinstance(raw, dict) and 'features' in raw:
        items = raw['features']
        feature_catalog = {str(it['id']): it for it in items}
    elif isinstance(raw, list):
        feature_catalog = {str(it['id']): it for it in raw}
    elif isinstance(raw, dict):
        feature_catalog = {str(k): v for k, v in raw.items()}
    print(f'feature_catalog.json loaded: {len(feature_catalog)} entries')
except (EntryNotFoundError, FileNotFoundError, Exception) as e:
    print(f'No feature_catalog.json in {HF_SAE_REPO} (that is fine — run 04 to get nice names).')
    print(f'  reason: {type(e).__name__}: {e}')
    feature_catalog = None

In [ ]:
# Cell 8 — generate tokens and capture residual @ LAYER for every generated token
import torch
from tqdm.auto import tqdm

# Hook captures the OUTPUT residual of layer LAYER. The output is either a Tensor or a
# tuple (hidden, ...) depending on architecture — we unwrap.
_captured = {'h': None}
def _hook(_mod, _inp, out):
    h = out[0] if isinstance(out, tuple) else out
    _captured['h'] = h.detach()

handle = layers[LAYER].register_forward_hook(_hook)

try:
    inputs = tok(PROMPT, return_tensors='pt').to(device)
    input_ids = inputs['input_ids']
    attn_mask = inputs.get('attention_mask', torch.ones_like(input_ids))
    prompt_len = input_ids.shape[1]

    generated_ids   = []    # list of int token ids
    residuals       = []    # list of [d_model] tensors (one per generated token)

    cur_ids, cur_mask = input_ids, attn_mask
    past = None

    for step in tqdm(range(MAX_NEW_TOKENS), desc='generate'):
        with torch.no_grad():
            out = model(
                input_ids=cur_ids,
                attention_mask=cur_mask,
                past_key_values=past,
                use_cache=True,
            )
        logits = out.logits
        past   = out.past_key_values

        # Residual of the *last* position (= the new token's representation).
        h = _captured['h']  # [B, T, d_model]
        residuals.append(h[0, -1].float().cpu())

        # Greedy next token.
        next_id = logits[0, -1].argmax().item()
        generated_ids.append(next_id)
        if tok.eos_token_id is not None and next_id == tok.eos_token_id:
            break

        cur_ids  = torch.tensor([[next_id]], device=device)
        cur_mask = torch.cat([cur_mask, torch.ones_like(cur_ids)], dim=1)
finally:
    handle.remove()

print(f'generated {len(generated_ids)} tokens, captured {len(residuals)} residual vectors')

# ---- SAE encode -------------------------------------------------------------
R = torch.stack(residuals, dim=0).to(device=device, dtype=torch.float32)  # [T_gen, d_model]
with torch.no_grad():
    feats = sae.encode(R)                      # [T_gen, d_sae]
feats = feats.cpu()
print(f'feature activation tensor: {tuple(feats.shape)}  ({feats.dtype})')

In [ ]:
# Cell 9 — pick top-10 features by summed activation, normalise each to [0, 1]
import torch

# Sum activation across tokens -> per-feature score -> top-N indices.
per_feat_total = feats.sum(dim=0)                          # [d_sae]
top_vals, top_idx = per_feat_total.topk(TOP_N_FEATURES)
top_idx = top_idx.tolist()

# Extract [N, T_gen] sub-matrix and normalise each row by its own max (avoid div-by-0).
sub  = feats[:, top_idx].T.contiguous()                     # [N, T_gen]
maxv = sub.amax(dim=1, keepdim=True).clamp_min(1e-8)
activations_norm = (sub / maxv).clamp(0.0, 1.0)             # in [0,1]

# Convert to plain lists of floats (rounded for JSON compactness).
activations_matrix = [
    [round(float(x), 4) for x in row.tolist()]
    for row in activations_norm
]

print(f'top-{TOP_N_FEATURES} feature ids (by total activation): {top_idx}')
print(f'activations matrix shape: [{len(activations_matrix)}][{len(activations_matrix[0])}]')

In [ ]:
# Cell 10 — assemble trace.json
import json

# ---- tokens (preserve leading-space semantics) -------------------------------
# Decode each id individually so ' patient' keeps its leading space.
def decode_tok(i):
    s = tok.decode([i], skip_special_tokens=False)
    # Drop explicit special tokens if the tokenizer leaked them.
    for st in (tok.bos_token, tok.eos_token, tok.pad_token, tok.unk_token):
        if st and s == st:
            return ''
    return s

tokens_out = [decode_tok(i) for i in generated_ids]
# Strip any empties produced by dropped specials without disturbing spaces elsewhere.
tokens_out = [t for t in tokens_out if t != '']

# Re-align activations if we dropped any tokens (rare; keeps shape consistent).
if len(tokens_out) != len(generated_ids):
    keep = [j for j, i in enumerate(generated_ids) if decode_tok(i) != '']
    activations_matrix = [[row[j] for j in keep] for row in activations_matrix]

# ---- features (with catalog lookup when available) --------------------------
def make_feature_entry(fid, rank):
    fid_s = str(fid)
    if feature_catalog and fid_s in feature_catalog:
        e = feature_catalog[fid_s]
        return {
            'id':    fid_s,
            'name':  e.get('name',  f'feature_{fid_s}'),
            'desc':  e.get('desc',  e.get('description', '')),
            'auroc': float(e.get('auroc', 0.0)),
        }
    # Fallback when catalog is missing — neutral values, downstream UI still renders.
    return {
        'id':    fid_s,
        'name':  f'feature_{fid_s}',
        'desc':  f'Feature #{fid_s} (run notebook 04 to label it).',
        'auroc': 0.0,
    }

features_out = [make_feature_entry(fid, r) for r, fid in enumerate(top_idx)]

# ---- final TraceData ---------------------------------------------------------
trace = {
    'prompt':          PROMPT,
    'model':           HF_BASE_MODEL,
    'layer':           f'L{LAYER} residual',
    'sae_repo':        HF_SAE_REPO,
    'tokens':          tokens_out,
    'features':        features_out,
    'activations':     activations_matrix,
    'counterfactuals': {},  # filled later by notebook 06 (steering)
}

# ---- light schema sanity check (catches shape drift early) -------------------
assert isinstance(trace['tokens'], list) and all(isinstance(t, str) for t in trace['tokens'])
assert isinstance(trace['features'], list) and len(trace['features']) == TOP_N_FEATURES
assert len(trace['activations']) == TOP_N_FEATURES
assert all(len(row) == len(trace['tokens']) for row in trace['activations'])
assert all(0.0 <= v <= 1.0 for row in trace['activations'] for v in row)
print('TraceData validated ✓')
print(f"  tokens     : {len(trace['tokens'])}")
print(f"  features   : {len(trace['features'])}")
print(f"  activations: [{len(trace['activations'])}][{len(trace['activations'][0])}]")

## 📤 Share your trace

You have two options — pick one or both:

**A. Upload to your SAE repo** (the cell below does this when `UPLOAD_TO_HF=True`). The file lives at
`https://huggingface.co/{HF_SAE_REPO}/resolve/main/trace.json` — anyone with the link can render it.

**B. Paste into the Trace Theater.** Open
[`openinterp.org/observatory/trace`](https://openinterp.org/observatory/trace) and drop the JSON into the
upload box, or use the `?src=hf://…/trace.json` query param once the fetch route ships (Q2 2026).

**Tip**: run [`06_steer_your_model.ipynb`](./06_steer_your_model.ipynb) next to populate
`counterfactuals` — it replays the same prompt with individual features clamped/ablated and writes the
alternate continuations back into `trace.json`. The Theater uses those for the feature hover-panel.

In [ ]:
# Cell 12 — save locally and (optionally) push to HF
import json
from pathlib import Path

out_path = Path(OUTPUT_TRACE)
with out_path.open('w') as f:
    json.dump(trace, f, ensure_ascii=False, indent=2)
print(f'wrote {out_path.resolve()}  ({out_path.stat().st_size:,} bytes)')

uploaded_url = None
if UPLOAD_TO_HF:
    try:
        from huggingface_hub import upload_file
        upload_file(
            path_or_fileobj=str(out_path),
            path_in_repo='trace.json',
            repo_id=HF_SAE_REPO,
            repo_type='model',
            commit_message=f'trace.json — prompt: {PROMPT[:60]!r}',
        )
        uploaded_url = f'https://huggingface.co/{HF_SAE_REPO}/resolve/main/trace.json'
        print(f'uploaded → {uploaded_url}')
    except Exception as e:
        print(f'upload skipped ({type(e).__name__}): {e}')
        print('  (local trace.json is still good — upload it manually or host it yourself)')

print()
print('─' * 72)
print('SHAREABLE LINKS')
print('─' * 72)
print(f'Local file : {out_path.resolve()}')
if uploaded_url:
    print(f'Raw JSON   : {uploaded_url}')
    print(f'Theater    : https://openinterp.org/observatory/trace?src=hf://{HF_SAE_REPO}/trace.json')
    print('             (the ?src= route ships Q2 2026 — for now, paste the JSON into the upload box)')
else:
    print('Theater    : https://openinterp.org/observatory/trace  (upload the local file)')
print('─' * 72)